# 47. 统计柱状图（barplot）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 4 / 20 步：比较类别频数、水平与组内分布**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 频数图（countplot）  →  **本章任务：** 统计柱状图（barplot）  →  **下一步：** 点图（pointplot）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

面对一张订单表，你关心的往往不是某一个具体订单，而是"哪个品类的客单价更高"、"哪个地区的销售额更大"这类把整组数据捏在一起看的问题。



## 本章目标

学完本章，你将能够：

- **理解**：理解「统计柱状图（barplot）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「统计柱状图（barplot）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「统计柱状图（barplot）」并读出其中的结论。


## 47.1 适用场景

**背景引入**：面对一张订单表，你关心的往往不是某一个具体订单，而是"哪个品类的客单价更高"、"哪个地区的销售额更大"这类把整组数据捏在一起看的问题。统计柱状图（barplot）正是为此而生——它把每一组的平均值或中位数画成一根柱子，再根据误差线告诉你这根柱子到底可不可靠。比起在一大串数字里横着比，几根柱子能更快地显露出组与组之间的差距和量级差异。

打个比方：barplot 像'给每组算平均分'——它把这组里几十上百个订单捏成一个平均线画成柱子，再补一条上下波动的误差线告诉你'这平均稳不稳'。柱子上的缺口并不代表某笔具体的单，而是这组数据的代表值。

比较各类别的平均值、中位数或自定义统计量。


## 47.2 数据结构

一列类别和一列数值；每组需要多个观察才能估计误差。


## 47.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 estimator="mean" 改为 estimator="median"，对比均值与中位数的柱高差异
2. 修改 errorbar=("ci", 90) 为 errorbar="sd"，观察置信区间与标准差的误差线长度
3. 将 errorbar=None 改为 errorbar=("ci", 95)，说明误差线对估计不确定性的表达作用


## 47.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `orders.groupby()`、`plt.subplots()`、`sns.barplot()`、`ax.set()` | 比较各类别的平均值、中位数或自定义统计量。 | 把均值柱高解释为总量 |
| 进阶变体 | `plt.subplots()`、`sns.barplot()`、`ax.set()`、`ax.legend()` | 在基础图表上增加分组、注释、布局或交互 | 隐藏分布和样本量 |
| 关键参数 | `estimator` | 估计量 | 把均值柱高解释为总量 |
| 关键参数 | `errorbar` | 误差表示 | 隐藏分布和样本量 |
| 关键参数 | `hue` | 分组 | 误差线含义不明确 |
| 关键参数 | `order` | 顺序 | 把均值柱高解释为总量 |


## 47.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-47 -->
### 数学推导｜均值的不确定性区间

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜样本均值存在抽样波动。** 独立同分布条件下 $\operatorname{Var}(\bar X)=\sigma^2/n$。

**第 2 步｜用样本标准差估计未知的 $\sigma$。** 得到 $SE\approx s/\sqrt n$。

**第 3 步｜用标准化分布给出区间。** 大样本近似下

$$
\frac{\bar X-\mu}{SE}\approx N(0,1)
$$

标准正态中约 95% 落在 $[-1.96,1.96]$，移项后得到 $\bar x\pm1.96SE$。小样本时应把 1.96 换成相应的 $t$ 分位数。

**把上面的关系收束为本章计算式：**

$$
CI_{95\%}\approx \bar{x}\pm1.96\frac{s}{\sqrt{n}}
$$

**符号解释：** $\bar{x}$ 是样本均值，$s$ 是样本标准差，$n$ 是样本量。

**代码对应：** 统计图中的误差线应明确表示标准差、标准误还是置信区间。

**使用边界：** 该近似依赖样本与分布条件；小样本或偏态数据可考虑 bootstrap。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(f"Diamonds {len(diamonds):,} | Taxis {len(taxis):,} | Flights {len(flights):,} 行")


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(f"样本：orders {len(orders):,} | marketing {len(marketing):,} | daily {len(daily):,} 行")


## 47.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


**练一练**：沿用 `orders` 里的 `category` 与 `order_value`，把基础图表的估计量从 `mean`（平均值）改成 `median`（中位数）再画一次柱状图。先想一想：均值更容易被极端高价拉高，换成中位数后，哪一类钻石的柱高下降最明显？填好下面三处 `____` 运行，再用右侧自检核对你的列名和估计结果。


In [ ]:
# 请在下方填写代码
# 目标：把估计量从 mean 改成 median，观察柱高的变化，并与左侧 summary 对照
import matplotlib.pyplot as plt
import seaborn as sns

# ==== 填空开始 ====
ESTIMATOR = "____"  # 请填写：绘图用的估计量（"mean" 或 "median"）
X_COL = "____"  # 请填写：横轴列名
Y_COL = "____"  # 请填写：纵轴（数值）列名
# ==== 填空结束 ====


In [ ]:
# 完整答案：把估计量改为 median，并与均值对比柱高差异。
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 4.2))
sns.barplot(
    data=orders,
    x="category",
    y="order_value",
    estimator="median",
    color="#1a73e8",
    ax=ax,
)
ax.set(title="品类客单价（中位数）", xlabel="品类", ylabel="客单价（元）")
plt.show()

# ---------- 计算结果对比（均值 vs 中位数） ----------
_mean = orders.groupby("category", sort=True)["order_value"].mean()
_median = orders.groupby("category", sort=True)["order_value"].median()
print("各类客单价均值：\n", _mean.round(1))
print("各类客单价中位数：\n", _median.round(1))
_diff = (_mean - _median).abs()
print("柱高变化最大的品类：", _diff.idxmax(), "，均值-中位数差值", round(_diff.max(), 1))


In [ ]:
import matplotlib.pyplot as plt

summary = (
    orders.groupby("category")["order_value"]
    .agg(["mean", "median", "count"])
    .round(1)
)
display(summary)
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.barplot(
    data=orders, x="category", y="order_value", ci=None, color="#1a73e8", ax=ax
)
ax.set(title="品类平均客单价", xlabel="品类", ylabel="平均客单价（元）")
fig.tight_layout()
plt.show()


## 47.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(
    data=orders,
    x="category",
    y="order_value",
    hue="channel",
    estimator="mean",
    errorbar=("ci", 90),
    palette="colorblind",
    ax=ax,
)
ax.set(title="分渠道比较品类客单价", xlabel="品类", ylabel="平均客单价（元）")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()


## 47.8 参数说明

- estimator：估计量
- errorbar：误差表示
- hue：分组
- order：顺序


## 47.9 结果解读

柱高是估计值，误差线含义由errorbar参数决定；同时报告样本量。


## 47.10 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 47.10.1 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 47.10.2 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 47.11 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 47.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 47.12 易错点提醒

- 把均值柱高解释为总量
- 隐藏分布和样本量
- 误差线含义不明确


## 47.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 47.14 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：把「平均客单价」换成「平均件数」指标，观察两种统计量差异
# 【目标】换一个 y 指标，看统计量的单位与含义如何变化。
import matplotlib.pyplot as plt
import seaborn as sns

# 起点示例(已可运行)：y 换成 items(购买件数)，看不同指标的分组均值。
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.barplot(data=orders, x="category", y="items", ci=None, color="#188038", ax=ax)
ax.set(title="品类平均购买件数", xlabel="品类", ylabel="平均件数")
fig.tight_layout()
plt.show()

# ---- 反思记录：换指标后，结论从「钱」变成什么 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
sns.barplot(
    data=orders,
    x="region",
    y="items",
    estimator="mean",
    errorbar="sd",
    color="#188038",
    ax=ax,
)
ax.set(title="区域平均购买件数及标准差", xlabel="区域", ylabel="件数")
fig.tight_layout()
plt.show()


## 47.15 小结

用barplot比较分类组的均值或其他估计量，并理解误差线。


### 47.15.1 你已经掌握

- 判断统计柱状图（barplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 47.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `estimator` | 估计量 |
| `errorbar` | 误差表示 |
| `hue` | 分组 |
| `order` | 顺序 |


### 47.15.3 需要注意

- 把均值柱高解释为总量
- 隐藏分布和样本量
- 误差线含义不明确


### 47.15.4 完成检查

- [ ] 能判断什么问题适合使用统计柱状图（barplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 47.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
